## 17. Macro Integration — FRED Data & Time-Varying Cox Model

In [ ]:
# ============================================================
# CELL 1
# ============================================================
!pip install fredapi --quiet

import os
from fredapi import Fred

FRED_API_KEY = os.environ.get("FRED_API_KEY")  # get a free key at https://fred.stlouisfed.org/docs/api/api_key.html
fred = Fred(api_key=FRED_API_KEY)

In [ ]:
# ============================================================
# CELL 2 — Pull macro series from FRED
# ============================================================
macro_series = {
    'unemployment_rate': 'UNRATE',
    'gdp_growth':        'A191RL1Q225SBEA',   # quarterly, will be forward-filled to monthly
    'treasury_10y':      'DGS10',
    'case_shiller_hpi':  'CSUSHPISA',
    'vix':               'VIXCLS',
}

macro_raw = {}
for name, code_ in macro_series.items():
    s = fred.get_series(code_)
    s.index = pd.to_datetime(s.index)
    macro_raw[name] = s
    print(f"{name:<20} ({code_}): {s.index.min().date()} to {s.index.max().date()}, {len(s)} obs")

In [ ]:
# ============================================================
# CELL 3 — Build a clean monthly macro panel
# ============================================================
macro_monthly = pd.DataFrame(index=pd.date_range('2006-01-01', '2019-06-01', freq='MS'))

for name, s in macro_raw.items():
    monthly = s.resample('MS').mean()
    monthly = monthly.reindex(macro_monthly.index).ffill()
    macro_monthly[name] = monthly

macro_monthly = macro_monthly.dropna()
print(f"Macro panel shape: {macro_monthly.shape}")
macro_monthly.tail()

In [ ]:
# ============================================================
# CELL 4 — Visual sanity check, recession period shaded red
# ============================================================
fig, axes = plt.subplots(len(macro_series), 1, figsize=(12, 14), sharex=True)
for ax, name in zip(axes, macro_series.keys()):
    ax.plot(macro_monthly.index, macro_monthly[name], color='#2c3e50')
    ax.axvspan(pd.Timestamp('2007-12-01'), pd.Timestamp('2009-06-01'), color='red', alpha=0.15)
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 5 — Macro value at origination (static building block)
# ============================================================
issue_dates = pd.to_datetime(df_cleaned['issue_d'], format='mixed')
issue_month_start = issue_dates.values.astype('datetime64[M]').astype('datetime64[ns]')

macro_at_origination = macro_monthly.reindex(issue_month_start).reset_index(drop=True)
macro_at_origination.index = df_cleaned.index

for col in macro_monthly.columns:
    df_cleaned[f'{col}_origination'] = macro_at_origination[col]

print("Macro-at-origination merge check:")
df_cleaned[[f'{c}_origination' for c in macro_monthly.columns]].describe()

In [ ]:
# ============================================================
# CELL 6 — Build the long-format (time-varying) sample
# ============================================================
from lifelines import CoxTimeVaryingFitter

N_SAMPLE = 30000  # start here; raise if runtime allows

tv_base = df_cleaned[cox_features + ['default']].copy()
tv_base['duration_months'] = survival_df['duration_months'].reindex(tv_base.index)
tv_base['issue_d'] = issue_dates.reindex(tv_base.index)
tv_base = tv_base.dropna(subset=cox_features + ['default', 'duration_months', 'issue_d'])

tv_sample = tv_base.sample(n=min(N_SAMPLE, len(tv_base)), random_state=42).reset_index(drop=True)
tv_sample['id'] = tv_sample.index

print(f"Building long-format panel for {len(tv_sample):,} loans...")

In [ ]:
# ============================================================
# CELL 7 — Reshape to long format (this is the slow cell)
# ============================================================
macro_lookup = macro_monthly['unemployment_rate']

long_rows = []
for _, row in tv_sample.iterrows():
    duration = int(row['duration_months'])
    issue = row['issue_d'].to_period('M').to_timestamp()
    for month in range(duration):
        current_date = (issue + pd.DateOffset(months=month)).to_period('M').to_timestamp()
        unrate = macro_lookup.get(current_date, np.nan)
        long_rows.append({
            'id': row['id'],
            'start': month,
            'stop': month + 1,
            'event': 1 if (month == duration - 1 and row['default'] == 1) else 0,
            'unemployment_rate': unrate,
            **{f: row[f] for f in cox_features}
        })

long_df = pd.DataFrame(long_rows).dropna()
print(f"Long-format rows: {len(long_df):,}  (from {len(tv_sample):,} loans)")

In [ ]:
# ============================================================
# CELL 8 — Fit the time-varying Cox model
# ============================================================
ctv = CoxTimeVaryingFitter(penalizer=0.1)
ctv.fit(long_df, id_col='id', start_col='start', stop_col='stop', event_col='event',
        show_progress=True)
ctv.print_summary()

In [ ]:
# ============================================================
# CELL 9 — Extract the unemployment hazard ratio
# ============================================================
unrate_coef = ctv.params_['unemployment_rate']
unrate_hr = np.exp(unrate_coef)

print(f"Unemployment coefficient (beta): {unrate_coef:.4f}")
print(f"Unemployment hazard ratio:       {unrate_hr:.4f}")
print(f"\nInterpretation: each 1-point rise in unemployment multiplies instantaneous "
      f"default hazard by {unrate_hr:.3f}x, holding borrower characteristics fixed.")

ctv.plot()
plt.title('Time-Varying Cox — Hazard Ratios incl. Macro')
plt.tight_layout()
plt.show()

**First attempt result:** the naive time-varying Cox model returns an unemployment hazard ratio of 0.96 — i.e. *higher* unemployment slightly *lowers* default hazard, holding borrower characteristics fixed. That's directionally implausible, so before trusting it, the next cell checks whether it's actually picking up unemployment's effect or just absorbing something else (origination-vintage drift).

In [ ]:
# ============================================================
# FIX — Add a vintage-year control to separate seasoning/underwriting
# drift from unemployment's actual effect
# ============================================================

# Re-use the same tv_sample from before (or re-sample if you want a fresh draw)
tv_sample['issue_year'] = tv_sample['issue_d'].dt.year

# Use a continuous "months since first loan" trend rather than a categorical
# year — this absorbs the steady vintage/underwriting drift as a straight-line
# trend, leaving unemployment's month-to-month wiggles for its own coefficient
# to capture.
tv_sample['vintage_trend'] = (
    (tv_sample['issue_year'] - tv_sample['issue_year'].min()) * 12 +
    (tv_sample['issue_d'].dt.month - 1)
)

print("Vintage trend range:", tv_sample['vintage_trend'].min(), "to", tv_sample['vintage_trend'].max())

In [ ]:
# ============================================================
# Rebuild long_rows with vintage_trend included as a covariate
# ============================================================
macro_lookup = macro_monthly['unemployment_rate']

long_rows = []
for _, row in tv_sample.iterrows():
    duration = int(row['duration_months'])
    issue = row['issue_d'].to_period('M').to_timestamp()
    for month in range(duration):
        current_date = (issue + pd.DateOffset(months=month)).to_period('M').to_timestamp()
        unrate = macro_lookup.get(current_date, np.nan)
        long_rows.append({
            'id': row['id'],
            'start': month,
            'stop': month + 1,
            'event': 1 if (month == duration - 1 and row['default'] == 1) else 0,
            'unemployment_rate': unrate,
            'vintage_trend': row['vintage_trend'],   # NEW — absorbs seasoning/underwriting drift
            **{f: row[f] for f in cox_features}
        })

long_df = pd.DataFrame(long_rows).dropna()
print(f"Long-format rows: {len(long_df):,}  (from {len(tv_sample):,} loans)")

In [ ]:
# ============================================================
# Refit the time-varying Cox model WITH the vintage control
# ============================================================
from lifelines import CoxTimeVaryingFitter

ctv_v2 = CoxTimeVaryingFitter(penalizer=0.1)
ctv_v2.fit(long_df, id_col='id', start_col='start', stop_col='stop', event_col='event',
           show_progress=True)
ctv_v2.print_summary()

In [ ]:
# ============================================================
# Extract and compare the corrected unemployment hazard ratio
# ============================================================
unrate_coef_v2 = ctv_v2.params_['unemployment_rate']
unrate_hr_v2 = np.exp(unrate_coef_v2)

vintage_coef = ctv_v2.params_['vintage_trend']
vintage_hr = np.exp(vintage_coef)

print(f"--- BEFORE (no vintage control) ---")
print(f"Unemployment hazard ratio: {unrate_hr:.4f}")
print()
print(f"--- AFTER (with vintage control) ---")
print(f"Unemployment hazard ratio: {unrate_hr_v2:.4f}")
print(f"Vintage trend hazard ratio (per month of platform age): {vintage_hr:.4f}")
print()
print(f"Interpretation: each 1-point rise in unemployment now multiplies default "
      f"hazard by {unrate_hr_v2:.3f}x, holding borrower characteristics AND vintage "
      f"drift fixed.")
print(f"The vintage_trend coefficient absorbs the steady seasoning/underwriting-drift "
      f"effect separately — {vintage_hr:.4f}x per additional month of platform age.")

ctv_v2.plot()
plt.title('Time-Varying Cox WITH Vintage Control — Hazard Ratios')
plt.tight_layout()
plt.show()

# ============================================================
# Use this corrected value going forward for stress testing
# ============================================================
unrate_hr = unrate_hr_v2   # overwrite — this is now the value the afternoon
                            # stress-testing block will use
print(f"unrate_hr updated to corrected value: {unrate_hr:.4f}")

### Diagnosing the identification problem

In [ ]:
# ============================================================
# DIAGNOSTIC 1 — How much does unemployment_rate actually vary
# within the sample the model is using?
# ============================================================
print("--- Unemployment rate distribution in long_df ---")
print(long_df['unemployment_rate'].describe())

print("\n--- Unemployment rate by issue year (in the sample) ---")
year_map = tv_sample.set_index('id')['issue_year']
long_df['issue_year'] = long_df['id'].map(year_map)
print(long_df.groupby('issue_year')['unemployment_rate'].agg(['mean', 'min', 'max', 'std']))

# If std is tiny and min/max range is narrow, the model has very little
# signal to learn an unemployment effect from at all.

In [ ]:
# ============================================================
# DIAGNOSTIC 2 — Raw relationship: unemployment vs default,
# no model, just direct comparison
# ============================================================

# Bucket loan-months by unemployment level, look at raw event (default) rate
long_df['unrate_bucket'] = pd.cut(long_df['unemployment_rate'],
                                    bins=[0, 4, 5, 6, 7, 8, 100],
                                    labels=['<4%', '4-5%', '5-6%', '6-7%', '7-8%', '8%+'])

print("--- Raw event rate by unemployment bucket (loan-month level) ---")
print(long_df.groupby('unrate_bucket').agg(
    event_rate=('event', 'mean'),
    n_months=('event', 'count')
))

# Also check: does this just mirror issue_year, i.e. is unrate_bucket basically
# identical to issue_year in disguise?
print("\n--- Cross-tab: issue_year vs unrate_bucket (loan-month level) ---")
print(pd.crosstab(long_df['issue_year'], long_df['unrate_bucket']))

In [ ]:
# ============================================================
# DIAGNOSTIC 3 — Is unemployment_rate collinear with int_rate or grade_woe?
# (i.e. is the model "stealing" unemployment's signal via a correlated feature)
# ============================================================

corr_check_cols = ['unemployment_rate', 'int_rate', 'grade_woe', 'vintage_trend']
print("--- Correlation matrix among these covariates (loan-month level) ---")
print(long_df[corr_check_cols].corr().round(3))

# A high correlation (e.g. |corr| > 0.5) between unemployment_rate and
# int_rate or grade_woe would mean the model can't cleanly tell them apart —
# whichever one "wins" the regression is partly arbitrary.

### Limitation and Resolution — Macro Sensitivity Could Not Be Reliably Estimated

A time-varying Cox model was fit to estimate the sensitivity of default hazard to the unemployment rate. The fitted hazard ratio (0.96) was directionally implausible — diagnostics showed origination vintage and unemployment rate are correlated at r = -0.755 in this sample, meaning the dataset lacks sufficient independent variation between calendar time and macro conditions to identify unemployment's effect separately from vintage/underwriting-quality drift (Lending Club's loan volume and underwriting standards changed substantially between 2012-2017, the same window over which unemployment fell steadily post-recession). Adding a vintage control did not resolve this, confirming the two variables are too collinear to disentangle in this sample.
Given this identification problem, the stress-testing exercise below uses an assumed hazard ratio of 1.10–1.15 per 1-point unemployment increase, consistent with the direction and approximate magnitude found in published consumer-credit research (e.g., Gerardi et al., Federal Reserve Bank of Atlanta, find unemployment raises default probability by 5-13 percentage points in mortgage data), rather than the dataset's own (unreliable) fitted coefficient. Notably, the Federal Reserve has also documented cases (COVID-19) where this historical relationship temporarily decoupled due to policy interventions — a reminder that even well-identified macro sensitivities are not guaranteed to hold under all conditions.